# Bike Availability Forecasting

This notebook trains several regression models to forecast bike availability at stations using historical usage, weather, calendar variables and tensor-based encodings.

The pipeline includes:
- Data loading
- Tensor decomposition based feature encoding
- Covariate construction
- Model training and evaluation

In [1]:
!pip install xarray scikit-learn xgboost

In [2]:
!pip install netcdf4

In [3]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------

import numpy as np
import xarray as xr

# Tensor decomposition
from tensorly.decomposition import tucker

# ML models
import xgboost as xgb

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

from utilities import To_Numpy, PolyReg, concat_features

# ---------------------------------------------------------
# Global reproducibility
# ---------------------------------------------------------

np.random.seed(42)

KeyboardInterrupt: 

BENCHMARKED MODELS:

In [ ]:
# ---------------------------------------------------------
# Model registry (benchmark definition)
# ---------------------------------------------------------

MODEL_REGISTRY = {
    # -----------------------------------------------------
    # Linear baselines
    # -----------------------------------------------------
    "lin": {
        "name": "LinearRegression",
        "model": LinearRegression,
        "params": {}
    },
    # -----------------------------------------------------
    # Gradient boosting
    # -----------------------------------------------------
    "xgb": {
        "name": "XGBoost",
        "model": xgb.XGBRegressor,
        "params": {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "max_depth": 8
        }
    },
    # -----------------------------------------------------
    # Neural network
    # -----------------------------------------------------
    "neural": {
        "name": "MLPRegressor",
        "model": MLPRegressor,
        "params": {
            "hidden_layer_sizes": (30, 30),
            "activation": "relu",
            "solver": "adam",
            "max_iter": 10,
            "random_state": 42
        }
    },
    # -----------------------------------------------------
    # Polynomial regression
    # -----------------------------------------------------
    "poly": {
        "name": "Polynomial",
        "model": PolyReg,
        "params": {
            "degree": 2,
            "ridge": 0.01,
        }
    }
}

LOAD DATA

In [ ]:
# Load dataset
ds = xr.open_dataset("../full_db.nc")
ds["bike"] = ds["vm_disp"] + ds["vae_disp"]

print(ds)

<xarray.Dataset> Size: 357MB
Dimensions:          (station: 1551, binary_index: 11, spatial_feature: 4,
                      time: 17544, ferie_feature: 2, component: 10, meteo_dim: 8)
Coordinates:
  * station          (station) <U12 74kB '000000002' '10001' ... '9999980'
    capacite         (station) int32 6kB ...
    latitude         (station) float64 12kB ...
    longitude        (station) float64 12kB ...
  * binary_index     (binary_index) int32 44B 0 1 2 3 4 5 6 7 8 9 10
  * spatial_feature  (spatial_feature) <U12 192B 'Capacité de ' ... 'altitude'
  * time             (time) datetime64[ns] 140kB 2023-01-01 ... 2024-12-31T23...
  * ferie_feature    (ferie_feature) <U8 64B 'vacances' 'is_ferie'
  * component        (component) int64 80B 0 1 2 3 4 5 6 7 8 9
  * meteo_dim        (meteo_dim) <U5 160B 'DRR1' 'FXI' 'GLO' ... 'TX' 'U'
Data variables: (12/20)
    code_binaire     (station, binary_index) int32 68kB ...
    features         (station, spatial_feature) float64 50kB ...
   

PIPELINE CONFIGURATION

In [ ]:
train_sample_sizes = [40_000, 200_000, 1_000_000]
horizons = [0, 24, 24 * 7]
model_list = ["lin","xgb", "neural", "poly"]
nsplits = 1

hnames = {h: f"{h} <= t < {h+24}" for h in horizons}
print("train_sizes:", train_sample_sizes)
print("horizons:", list(hnames.values()))
print("models:", model_list)
print("pipeline should run in < 10 minutes")

train_sizes: [40000, 200000, 1000000]
horizons: ['0 <= t < 24', '24 <= t < 48', '168 <= t < 192']
models: ['lin', 'xgb', 'neural', 'poly']
pipeline should run in < 10 minutes


TIME CONFIGURATION

In [ ]:
# First Monday of 2023 (dataset starts on a Sunday)
first_monday_index = 24

n_stations = len(ds["station"])

n_time_full = len(ds["time"]) - first_monday_index

n_weeks = n_time_full // (7 * 24)

n_time = 24 * 7 * n_weeks
n_days = 7 * n_weeks

# ---------------------------------------------------------
# Static covariates
# ---------------------------------------------------------
#memory efficiency
float_tp = np.float32
#Xarray -> Numpy
to_numpy = To_Numpy(ds, first_monday_index, n_weeks, float_tp)

#/!\ a few station capacity are negative due to a preprocessing bug
capacity = np.positive(to_numpy("capacite"))

lat = to_numpy("latitude")
lon = to_numpy("longitude")

Meteo = to_numpy("meteo_h")
Holiday = to_numpy("feries")

IMPORTANT: EMBEDDING FUNCTION (with Tucker)

In [ ]:
rank = [10, n_weeks, 7, 10]

def tensor_encoding(x):
    """
    Extract spatial and temporal embeddings using Tucker decomposition.

    Dimensions:
    station × week × day × hour
    """

    _, factors = tucker(
        x.reshape(n_stations, n_weeks, 7, 24).astype(np.float32),
        rank=rank
    )

    spatial_encoding = (factors[0] / (capacity[:, None] + 0.01))

    day_encoding = factors[2]
    hour_encoding = factors[3]

    time_encoding = concat_features(
        [day_encoding[:, None, :], hour_encoding[None, :, :]],
        shape=[7, 24],
        flatten=True
    )

    return concat_features([capacity[:, None], spatial_encoding]).astype(float_tp), time_encoding.astype(float_tp)

MODEL FIT

In [ ]:
def fit_model(train_test, model_key):
    """
    Train model defined in MODEL_REGISTRY and return predictions.
    """

    X_train, X_test, y_train, y_test = train_test
    X_train = X_train.copy()
    X_test = X_test.copy()

    config = MODEL_REGISTRY[model_key]

    # -----------------------------------------------------
    # Model instantiation
    # -----------------------------------------------------

    model = config["model"](**config["params"])

    # -----------------------------------------------------
    # Training
    # -----------------------------------------------------

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    rmse = np.sqrt(np.mean((pred - y_test) ** 2))

    print(f"{model_key} RMSE: {rmse:.3f}")

    return pred, model

## Covariates used in the model

- Lagged bike counts
- Meteorological variables
- Holiday indicators
- Spatial station embeddings
- Temporal embeddings (day/hour)
- Weekly seasonal encoding
- Last known daily update

In [ ]:
# ---------------------------------------------------------
# Cross-validation split across weeks
# ---------------------------------------------------------

# First 10 weeks cannot be used (lags require history)
valid_weeks = np.arange(10, n_weeks)

# 4-fold temporal split
fold_index = (valid_weeks - 10) % 4

weeks_split = [
    [valid_weeks[fold_index != i], valid_weeks[fold_index == i]]
    for i in range(4)
]

# ---------------------------------------------------------
# Seasonal week encoding
# ---------------------------------------------------------

def week_encoding(week_index):
    """
    Encode week index using Fourier seasonal features.
    """
    harmonics = []

    for h in [1, 2, 3]:
        harmonics.append(np.sin(2 * np.pi * h * week_index / 52))
        harmonics.append(np.cos(2 * np.pi * h * week_index / 52))

    return np.column_stack([week_index, *harmonics]).astype(float_tp)


# ---------------------------------------------------------
# Lag utilities
# ---------------------------------------------------------

def lag(array, lag_value):
    """
    Temporal lag operator.
    """
    return np.roll(array, shift=lag_value)


# Weekly historical lags
week_lags = [24 * 7 * i for i in range(1, 10)]


# ---------------------------------------------------------
# Last observed daily value
# ---------------------------------------------------------

def last_daily_update(Y, horizon):
    """
    Last known station value at 23:00 before prediction horizon.
    """

    daily_last = (
        Y.reshape(n_stations, n_weeks, 7, 24)[:, :, :, -1]
        .reshape(n_stations, n_weeks, 7)
    )

    return lag(daily_last, horizon // 24 + 1)

In [ ]:
def prepare_data(variable):
    """
    Prepare base tensors and embeddings for a given variable.
    """

    Y = to_numpy(variable)

    spatial_features, temporal_features = tensor_encoding(Y)

    spatial_features = np.concatenate(
        [spatial_features, lat[:, None], lon[:, None]],
        axis=1
    )

    return {
        "Y": Y,
        "space": spatial_features,
        "time": temporal_features
    }

PREPARE DATA FOR REGRESSION TASK

In [ ]:
#prepare target embedding (Tucker)
data_dict = prepare_data("bike")

In [ ]:
def prepare_split_horizon(split_id, horizon, data_dict, samplers):

    Y = data_dict["Y"]
    space = data_dict["space"]
    time = data_dict["time"]

    week_splits = {"train": weeks_split[split_id][0], "test": weeks_split[split_id][1]}
    # Lag configuration
    lags = [24 * i + horizon for i in range(1, 10)] + week_lags

    Xy = {"X": {}, "y": {}}

    for mode in ["train", "test"]:
        sampler = samplers[mode]
        stations, weeks, tows,  = sampler(n_stations), sampler(week_splits[mode]), sampler(24*7)
        #========
        #Target Y
        Xy["y"][mode] = Y[stations, weeks, tows]
        #========
        #Features
        #lags
        Ylag = concat_features([
            lag(Y, l)[stations, weeks, tows][:, None]
            for l in lags
        ])
        #last update
        Ylast = last_daily_update(Y, horizon)[stations, weeks, tows//24][:, None]

        #merge features
        Xy["X"][mode]= concat_features([
            Ylag,
            Meteo[weeks, tows],
            Holiday[weeks, tows],
            space[stations],
            time[tows],
            week_encoding(weeks),
            Ylast
        ])

    return (Xy["X"]["train"], Xy["X"]["test"], Xy["y"]["train"], Xy["y"]["test"])

TRAINING LOOP

In [ ]:
# ---------------------------------------------------------
# Sampling utility for large datasets
# ---------------------------------------------------------
for train_size in train_sample_sizes:

    print("=======================")
    print("train size:", train_size)
    def sampler_train(x):
        return np.random.choice(x, size=train_size)
    def sampler_test(x):
        return np.random.choice(x, size=1_000_000)
    samplers = {"train": sampler_train, "test": sampler_test}

    for split_id in range(nsplits):

        for horizon_id, horizon in enumerate(horizons):
            print("=======================")
            print("horizon:", hnames[horizon])

            train_test = prepare_split_horizon(
                split_id,
                horizon,
                data_dict,
                samplers
            )

            X_train, X_test, y_train, y_test = train_test

            scaler = StandardScaler()

            X_train = scaler.fit_transform(X_train).astype(float_tp)
            X_test = scaler.transform(X_test).astype(float_tp)

            train_test = (
                np.nan_to_num(X_train),
                np.nan_to_num(X_test),
                y_train.astype(float),
                y_test
            )

            for model_name in model_list:
                pred, model = fit_model(
                    train_test,
                    model_name
                )

                #save interaction matrix build on polynomial model
                if [train_size, horizon] == [train_sample_sizes[-1], horizons[0]]:
                  if model_name == "poly":
                    #save interactions
                    np.save("interactions.npy", model.get_interactions())
                    print("interaction matrix saved !")
                  if model_name == "xgb":
                    #save feature importance
                    np.save("feature_importance.npy", model.feature_importances_)
                    print("feature importance saved !")

Covariate indexes:
(1:18 lag)
(19:26 Meteo)
(27:28 Holiday)
(29 capacity)
(30:39 Space embedding)
(40:41 lat lon)
(42:48 day embedding)
(49:58 hour embedding)
(59:65 week embedding)
(66: last update)

In [ ]:
covariates = {

    "lag": (
        "Hourly lags [24*(i+DH)] up to 9 days where DH is the daily forecast horizon"
        "+ weekly lags [24*7*i] up to 9 weeks"
    ),

    "meteo": ds.meteo_dim.values,

    "holiday": ds.ferie_feature.values,

    "space": (
        "Station capacity + "
        f"{rank[0]} spatial Tucker components + "
        "latitude / longitude"
    ),

    "time_of_week": (
        f"{rank[2]} day-of-week components "
        f"+ {rank[3]} hour-of-day components"
    ),

    "week": (
        "Absolute week index since 2023 "
        "+ 3 seasonal Fourier harmonics"
    ),

    "last_update": (
        "Last known station value "
        "at 23:00 before prediction horizon"
    )
}

In [ ]:
print(covariates)